<a href="https://colab.research.google.com/github/yf591/llm-toolkit/blob/main/gemma-4_llama-4_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma 4 & Llama 4 テキスト生成・チャットボット

このノートブックは、Hugging Faceで公開されているLLM（**Google Gemma 4** や **Meta Llama 4** など）を Google Colab 上で手軽に試すためのツールです。

### 主な機能
- **最新モデルへの対応**<br/>
`transformers` ライブラリを最新版にアップデートし、Gemma 4 等の新しいアーキテクチャを利用可能にします。
- **2つのモード**
    1. **一問一答形式**<br/>
        - システムプロンプトで役割を与え、単発の回答を生成します。
    2. **チャット形式（推奨）**<br/>
        - 過去の会話履歴を保持し、連続した対話が可能です。
- **操作性**<br/>
Gradio を使用したGUIにより、プログラミングの知識がなくてもパラメータ（最大トークン長など）を調整して生成を試せます。

### 使い方
1. 最初のセルで必要なライブラリをインストールします。
2. **「ランタイム」->「セッションを再起動」** を実行します（※重要）。
3. モデル（デフォルトは `gemma-4-E2B-it`）をロードします。
4. 用途に合わせて「一問一答」または「チャット(推奨)」のセルを実行し、表示されるURLから操作してください。

# 注意
以下のいずれかを選択して実行してください。
1. テキスト生成をおこなう場合（一問一答形式）
2. チャットを行う場合 <--- **こっちを推奨**

※ 「2. チャットを行う場合」を選択する場合は`google/gemma-4-E2B-it`のように末尾に`-it`や`-instruct`（Instructチューニング済み）などがついているモデルを使用することをオススメします。

## 必要なライブラリーのインストールおよびインポート

In [ ]:
from google.colab import output

# PyTorchとtorchvisionのバージョン不整合を解消し、Gemma 4に対応させます
!pip install transformers -U torch torchvision gradio

output.clear()
print('インストールが完了しました。メニューの「ランタイム」->「セッションを再起動」を実行してから、次のセルに進んでください。')

インストールが完了しました。メニューの「ランタイム」->「セッションを再起動」を実行してから、次のセルに進んでください。


In [ ]:
import torch  # PyTorchライブラリをインポート（テンソル演算、GPU使用などに必要）
from transformers import AutoModelForCausalLM, AutoTokenizer # Hugging Face Transformersライブラリから必要なクラスをインポート（言語モデル、トークナイザー）
import gradio as gr # Gradioライブラリをインポート（Web UI作成用）

In [ ]:
# Check GPU availability
!nvidia-smi

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Current Device: {torch.cuda.get_device_name(0)}")

## ModelとTokenizerをLoad

In [ ]:
# モデルとトークナイザーをロード

#@markdown ### ◆使用するモデルの名前 (Hugging FaceのモデルID)
model_name = "google/gemma-4-E2B-it" #@param ["google/gemma-4-31B-it", "google/gemma-4-31B", "google/gemma-4-26B-A4B-it", "google/gemma-4-26B-A4B", "google/gemma-4-E4B-it", "google/gemma-4-E4B", "google/gemma-4-E2B-it", "google/gemma-4-E2B", "meta-llama/Llama-4-Scout-17B-16E-Instruct", "meta-llama/Llama-4-Scout-17B-16E", "meta-llama/Llama-4-Maverick-17B-128E-Instruct", "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8", "meta-llama/Llama-4-Maverick-17B-128E", "meta-llama/Llama-4-Scout-17B-16E-Instruct-Original", "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8-Original", "meta-llama/Llama-4-Scout-17B-16E-Original", "meta-llama/Llama-4-Maverick-17B-128E-Instruct-Original", "meta-llama/Llama-4-Maverick-17B-128E-Original", "meta-llama/Llama-Prompt-Guard-2-22M", "meta-llama/Llama-Prompt-Guard-2-86M", "meta-llama/Llama-Guard-4-12B"] {type:"string"}

# トークナイザーをロード
tokenizer = AutoTokenizer.from_pretrained(model_name)

# モデルをロード
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

#@markdown ---
#@markdown **※補足：torch_dtype=torch.float16 について**
#@markdown float16を使用することでメモリ使用量を削減します。

#@markdown ---
#@markdown **※補足：device_map="auto" について**
#@markdown GPUが利用可能な場合は自動的にGPUが選択されます。

`model_name` を変更することで、Hugging Faceにある最新の言語モデルを簡単に試すことができます。

例えば、

1. **Gemma 4系 (Google)**
   - "google/gemma-4-31B-it"
   - "google/gemma-4-E2B"

2. **Llama 4系 (Meta)**
   - "meta-llama/Llama-4-Scout-17B-16E-Instruct"
   - "meta-llama/Llama-4-Maverick-17B-128E"
   - "meta-llama/Llama-Guard-4-12B"

3. **その他の多様なモデル**
   - "google/flan-t5-large"
   - "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
   - "HuggingFaceH4/zephyr-7b-beta"

**主な注意点**:
- **アーキテクチャの互換性**: Gemma 4やLlama 4のような最新モデルを利用する場合、`transformers` ライブラリが最新である必要があります。
- **GPUメモリ要件**: パラメータ数やモデルの形式（FP8など）によって必要なVRAM量が変わります。
- **トークナイザー**: モデルごとに固有の特殊トークンが存在するため、必ず対応する `AutoTokenizer` を使用してください。

**基本的なコードフレームワークは共通なので、`model_name` を変更してセルを再実行するだけで新しいモデルを試せます。**

## 1. テキスト生成をおこなう場合（一問一答形式）
※ 「2. チャットを行う場合」を実行する場合はこれは実行しない。<br/>
※ どちらかと言えばこのセクションの実行は非推奨で「2. チャットを行う場合」を実行することをお勧めする。そのため、このセクションのコードはコメントアウトしているので、必要であればコメントアウトを外して実行してください。

In [ ]:
# def generate_text(system_prompt, prompt, max_input_length, max_output_length):
#     """テキスト生成を行う関数

#     Args:
#         prompt (str): 入力テキスト文字列
#         max_input_length (int): 入力テキストの最大長
#         max_output_length (int): 生成するテキストの最大長

#     Returns:
#         str: 生成されたテキスト文字列
#     """

#     try:
#         # 1. プロンプトの構築（手動で分かりやすい形式を作っておく）
#         if system_prompt:
#             fallback_text = f"【システムからの指示】\n{system_prompt}\n\n【ユーザーからの入力】\n{prompt}\n\n【AIの回答】\n"
#         else:
#             fallback_text = f"【ユーザーからの入力】\n{prompt}\n\n【AIの回答】\n"

#         messages = [{"role": "user", "content": fallback_text}]

#         # 2. チャットテンプレートの適用を試みる
#         try:
#             # トークナイザーがチャットテンプレートを持っているかチェック
#             if not hasattr(tokenizer, 'chat_template') or tokenizer.chat_template is None:
#                 raise ValueError("No chat template") # 持っていない場合は意図的に下のexceptへ飛ばす

#             prompt_formatted = tokenizer.apply_chat_template(
#                 messages,
#                 tokenize=False,
#                 add_generation_prompt=True
#             )
#         except:
#             # テンプレートを持たない「ベースモデル」が選ばれた場合は、自作したテキストをそのまま使う
#             prompt_formatted = fallback_text

#         # 3. トークン化
#         inputs = tokenizer(
#             prompt_formatted,
#             truncation=True,
#             max_length=int(max_input_length),
#             return_tensors="pt"
#         ).to(model.device)

#         # 4. モデルによる生成
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=int(max_output_length),
#             do_sample=True,
#             temperature=0.7,
#             top_p=0.9,
#             repetition_penalty=1.3, # 繰り返しにペナルティを与える（1.0=なし、大きいほど強い）
#             no_repeat_ngram_size=4, # 同じ4単語の並びを繰り返さない
#             eos_token_id=tokenizer.eos_token_id,  # 終了トークンを明示
#             pad_token_id=tokenizer.eos_token_id,  # パディングも同じに
#         )

#         # 5. 生成された部分のみを切り出してデコード
#         input_length = inputs["input_ids"].shape[1]
#         generated_tokens = outputs[0][input_length:]

#         return tokenizer.decode(generated_tokens, skip_special_tokens=True)

#     except Exception as e:
#         return f"🚨 エラーが発生しました（🥺ぴえん）:\n{str(e)}"

In [ ]:
#@title GUIの設定（一問一答版）

# interface = gr.Interface(
#     fn=generate_text,
#     inputs=[
#         # システムプロンプト用のテキストボックス
#         gr.Textbox(
#             label="System Prompt (役割の設定)",
#             placeholder="（例）あなたは優秀なPythonエンジニアです。親しみやすい口調で、コードの解説を交えて答えてください。",
#             lines=3,
#         ),
#         gr.Textbox(
#             label="Prompt",
#             lines=10,
#             max_lines=20
#         ),
#         gr.Slider(
#             minimum=128,
#             maximum=8192,
#             value=4096,
#             step=1,
#             label="Max Input Length"
#         ),
#         gr.Slider(
#             minimum=128,
#             maximum=4096,
#             value=1024,
#             step=1,
#             label="Max Output Length"
#         )
#     ],
#     outputs=gr.Textbox(
#         label="Generated Text",
#         lines=15,
#         max_lines=40
#     ),
#     title="Custom LLM Chat",
#     description="システムプロンプトでAIの役割を定義し、LLMの出力をテストできます。"
# )

In [ ]:
# @title GUIを起動
# interface.launch(share=True, debug=True) # Gradioインターフェースを起動。share=Trueで外部からアクセス可能にする

## 2. チャットを行う場合
※ 「1. テキスト生成をおこなう場合（一問一答形式）」を実行する場合はこれは実行しない。<br/>
※ `google/gemma-4-E2B-it`のように末尾に`-it`（Instructチューニング済み）がついているモデルを使用する場合。

In [ ]:
# チャット対応のテキスト生成関数
def chat_with_llm(message, history, system_prompt, max_input_length, max_output_length):
    """
    記憶を保持するチャット用関数
    - message: 今回ユーザーが入力した新しいテキスト
    - history: 過去のやり取りのリスト [["ユーザーの質問1", "AIの回答1"], ["質問2", "回答2"]...]
    """
    try:
        # 1. メッセージリストの構築
        messages = []

        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})

        for item in history:
            if isinstance(item, dict):
                # Gradio の仕様（辞書型）
                messages.append({"role": item.get("role", "user"), "content": item.get("content", "")})
            elif isinstance(item, (list, tuple)):
                # Gradio 以前の仕様（リスト型）
                messages.append({"role": "user", "content": item[0]})
                messages.append({"role": "assistant", "content": item[1]})

        # 新しいメッセージを追加
        messages.append({"role": "user", "content": message})

        # 2. チャットテンプレートの適用
        try:
            if not hasattr(tokenizer, 'chat_template') or tokenizer.chat_template is None:
                raise ValueError("No chat template")

            prompt_formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except:
            # テンプレート非対応（ベースモデル）用のフォールバックもハイブリッド化
            prompt_formatted = ""
            if system_prompt:
                prompt_formatted += f"【指示】\n{system_prompt}\n\n"

            for item in history:
                if isinstance(item, dict):
                    role_str = "【ユーザー】" if item.get("role") == "user" else "【AI】"
                    prompt_formatted += f"{role_str}\n{item.get('content', '')}\n\n"
                elif isinstance(item, (list, tuple)):
                    prompt_formatted += f"【ユーザー】\n{item[0]}\n\n【AI】\n{item[1]}\n\n"

            prompt_formatted += f"【ユーザー】\n{message}\n\n【AI】\n"

        # 3. トークン化と長さの切り詰め
        inputs = tokenizer(
            prompt_formatted,
            truncation=True,
            max_length=int(max_input_length),
            return_tensors="pt"
        ).to(model.device)

        # 4. 生成
        outputs = model.generate(
            **inputs,
            max_new_tokens=int(max_output_length),
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3, # 繰り返しにペナルティを与える（1.0=なし、大きいほど強い）
            no_repeat_ngram_size=4, # 同じ4単語の並びを繰り返さない
            eos_token_id=tokenizer.eos_token_id,  # 終了トークンを明示
            pad_token_id=tokenizer.eos_token_id,  # パディングも同じに
        )

        # 5. デコード
        input_length = inputs["input_ids"].shape[1]
        generated_tokens = outputs[0][input_length:]

        return tokenizer.decode(generated_tokens, skip_special_tokens=True)

    except Exception as e:
        return f"🚨 エラーが発生しました（🥺ぴえん）:\n{str(e)}"

In [ ]:
# @title GUIの設定（チャット特化版）

# GUIの設定（ChatInterfaceを使用）
interface = gr.ChatInterface(
    fn=chat_with_llm,
    # System Promptやスライダーは「追加オプション」として折りたたみメニューに入れます
    additional_inputs=[
        gr.Textbox(
            label="System Prompt (役割の設定)",
            lines=3,
        ),
        gr.Slider(
            minimum=128, maximum=8192, value=4096, step=1,
            label="Max Input Length (過去の記憶を含めた最大文字数)"
        ),
        gr.Slider(
            minimum=128, maximum=4096, value=1024, step=1,
            label="Max Output Length (AIの返答の最大文字数)"
        )
    ],
    title="Custom LLM Chatbot",
    description="チャットアプリです。",
)

In [ ]:
# @title GUIを起動
interface.launch(share=True, debug=True) # Gradioインターフェースを起動。share=Trueで外部からアクセス可能にする